In [1]:
import optuna.exceptions
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Load Data

In [3]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
dataset_path = './dataset'

dataset = datasets.ImageFolder(root=dataset_path, transform=image_transforms)
len(dataset)

2300

In [5]:
dataset.classes

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [6]:
num_classes = len(dataset.classes)
num_classes

6

In [7]:
train_size = int(0.75*len(dataset))
val_size = len(dataset) - train_size

train_size, val_size

(1725, 575)

In [8]:
from torch.utils.data import random_split

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True, pin_memory=True)

## Model 5: ResNet with Hyper parameter tuning

In [10]:
class CarClassifierResNet(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.5):
        super().__init__()
        self.model = models.resnet50(weights='DEFAULT')
        # Freeze all layers except the final fully connected layer
        for param in self.model.parameters():
            param.requires_grad = False
        # Unfreeze layer 4 and fc layers
        for param in self.model.layer4.parameters():
            param.requires_grad = True
        # Replace the final fully connected layer
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.model.fc.in_features, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [11]:
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)

    # load the model
    model = CarClassifierResNet(num_classes=num_classes, dropout_rate=dropout_rate).to(device)

    # define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

    # training the loop
    epochs = 3
    for epoch in range(epochs):
        model.train()
        scaler = torch.amp.GradScaler('cuda')
        running_loss = 0.0

        for batch_num, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)
            # Zero the parameter gradients
            optimizer.zero_grad()
            # Forward pass
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            # Backward pass
            scaler.scale(loss).backward()
            # Update the weights
            scaler.step(optimizer)
            scaler.update()

            if (batch_num+1) % 10 == 0:
                print(f"Batch: {batch_num+1}, Epoch: {epoch+1}, Loss: {loss.item():.2f}")

            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss/len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

    # Model Evaluation:

        model.eval()
        total = 0.0
        correct = 0.0
        with torch.no_grad():

            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total

        trial.report(accuracy, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy

In [12]:
import optuna

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2026-09-30 21:29:32,057] A new study created in memory with name: no-name-60af4125-f237-4be8-a0f5-876d211bb06a


Batch: 10, Epoch: 1, Loss: 1.03
Batch: 20, Epoch: 1, Loss: 0.61
Batch: 30, Epoch: 1, Loss: 1.10
Batch: 40, Epoch: 1, Loss: 0.70
Batch: 50, Epoch: 1, Loss: 1.00
Epoch [1/3], Loss: 0.8635
Batch: 10, Epoch: 2, Loss: 0.70
Batch: 20, Epoch: 2, Loss: 0.34
Batch: 30, Epoch: 2, Loss: 0.29
Batch: 40, Epoch: 2, Loss: 0.61
Batch: 50, Epoch: 2, Loss: 0.53
Epoch [2/3], Loss: 0.4748
Batch: 10, Epoch: 3, Loss: 0.46
Batch: 20, Epoch: 3, Loss: 0.55
Batch: 30, Epoch: 3, Loss: 0.52
Batch: 40, Epoch: 3, Loss: 0.45
Batch: 50, Epoch: 3, Loss: 0.41
Epoch [3/3], Loss: 0.3622


[I 2026-09-30 21:33:41,962] Trial 0 finished with value: 77.04347826086956 and parameters: {'lr': 0.001322951703761863, 'dropout_rate': 0.5693955848336998}. Best is trial 0 with value: 77.04347826086956.


Batch: 10, Epoch: 1, Loss: 1.02
Batch: 20, Epoch: 1, Loss: 0.85
Batch: 30, Epoch: 1, Loss: 0.64
Batch: 40, Epoch: 1, Loss: 0.62
Batch: 50, Epoch: 1, Loss: 0.79
Epoch [1/3], Loss: 0.8912
Batch: 10, Epoch: 2, Loss: 0.36
Batch: 20, Epoch: 2, Loss: 0.44
Batch: 30, Epoch: 2, Loss: 0.50
Batch: 40, Epoch: 2, Loss: 0.54
Batch: 50, Epoch: 2, Loss: 0.41
Epoch [2/3], Loss: 0.4925
Batch: 10, Epoch: 3, Loss: 0.53
Batch: 20, Epoch: 3, Loss: 0.34
Batch: 30, Epoch: 3, Loss: 0.25
Batch: 40, Epoch: 3, Loss: 0.22
Batch: 50, Epoch: 3, Loss: 0.68
Epoch [3/3], Loss: 0.3422


[I 2026-09-30 21:36:55,705] Trial 1 finished with value: 81.04347826086956 and parameters: {'lr': 0.002165192319101128, 'dropout_rate': 0.44234944801118425}. Best is trial 1 with value: 81.04347826086956.


...

[I 2026-09-30 22:06:27,907] Trial 17 pruned. 


Batch: 10, Epoch: 1, Loss: 1.09
Batch: 20, Epoch: 1, Loss: 0.66
Batch: 30, Epoch: 1, Loss: 0.73
Batch: 40, Epoch: 1, Loss: 0.52
Batch: 50, Epoch: 1, Loss: 1.00
Epoch [1/3], Loss: 0.9140


[I 2026-09-30 22:07:36,458] Trial 18 pruned. 


Batch: 10, Epoch: 1, Loss: 0.93
Batch: 20, Epoch: 1, Loss: 0.85
Batch: 30, Epoch: 1, Loss: 0.75
Batch: 40, Epoch: 1, Loss: 0.80
Batch: 50, Epoch: 1, Loss: 0.70
Epoch [1/3], Loss: 0.8799


[I 2026-09-30 22:08:44,533] Trial 19 pruned. 


In [13]:
study.best_params

{'lr': 0.002165192319101128, 'dropout_rate': 0.44234944801118425}